# Customer Insights pipeline: Excel → Python → PostgreSQL

**What this notebook does**
1. Loads the dirty Excel file
2. Cleans Surveys, Complaints and Flights with Pandas
3. Shows a data-quality log (what was fixed)
4. Loads the clean tables into PostgreSQL (schema `pipeline`)

Run the cells **one by one** from top to bottom (Shift + Enter).

## Step 0: settings
Change the **file path** and your **PostgreSQL password**, then run.

In [ ]:
# First time only: install the libraries (remove the # and run once)
# %pip install pandas openpyxl sqlalchemy psycopg2-binary

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

FILE = "airline_cx_dataset_raw.xlsx"   # same folder as this notebook

DB_USER = "postgres"
DB_PASSWORD = "your_password"   # <-- your PostgreSQL password (never commit the real one)
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "qatar_kpi"

log = []   # data-quality log: one row per fix
def note(table, issue, rows, action):
    log.append({"table": table, "issue": issue, "rows": int(rows), "action": action})

## Step 1: load the three sheets
The real headers are on row 2 (row 1 is a title), so we use `header=1`.

In [ ]:
surveys    = pd.read_excel(FILE, sheet_name="Surveys",    header=1)
complaints = pd.read_excel(FILE, sheet_name="Complaints", header=1)
flights    = pd.read_excel(FILE, sheet_name="Flights",    header=1)

print("Surveys:", surveys.shape, "| Complaints:", complaints.shape, "| Flights:", flights.shape)
surveys.head()

## Step 2a: clean Surveys

In [ ]:
s = surveys.copy()
s.columns = ["survey_id", "customer_id", "survey_date", "route", "cabin_class",
             "csat_score", "nps_score", "ces_score", "issue_type"]

# 1. Duplicates on the ID (never on customer_id: repeat customers are real)
n = s.duplicated("survey_id").sum()
s = s.drop_duplicates("survey_id")
note("Surveys", "Duplicate SurveyID", n, "Removed")

# 2. Route: remove extra spaces
n = (s["route"] != s["route"].str.strip()).sum()
s["route"] = s["route"].str.strip()
note("Surveys", "Route with extra spaces", n, "Trimmed")

# 3. Cabin class: one spelling per class
cabin_map = {"economy": "Economy", "eco": "Economy", "economy class": "Economy",
             "business": "Business", "business class": "Business", "first": "First"}
before = s["cabin_class"].copy()
s["cabin_class"] = s["cabin_class"].str.strip().str.lower().map(cabin_map)
note("Surveys", "Cabin class spelling variants", (before != s["cabin_class"]).sum(), "Standardised")

# 4. Scores outside the valid range -> null (not guessed)
for col, lo, hi in [("csat_score", 1, 5), ("nps_score", 0, 10), ("ces_score", 1, 7)]:
    bad = ~s[col].between(lo, hi) & s[col].notna()
    note("Surveys", f"{col} outside {lo}-{hi}", bad.sum(), "Set to null")
    s.loc[bad, col] = np.nan
    note("Surveys", f"{col} blank", s[col].isna().sum() - bad.sum(), "Kept as null")

# 5. Issue type: one spelling per issue
issues = ["Baggage Delay", "Check-in", "Food & Beverage", "IFE System",
          "Lounge Access", "Refund", "Seat/Comfort", "Staff Behaviour"]
issue_map = {i.lower(): i for i in issues}
before = s["issue_type"].copy()
s["issue_type"] = s["issue_type"].str.strip().str.lower().map(issue_map).fillna("Unknown")
note("Surveys", "Issue type spelling variants", (before != s["issue_type"]).sum(), "Standardised")

# 6. Dates (dd/mm/yyyy text -> real dates)
s["survey_date"] = pd.to_datetime(s["survey_date"], format="%d/%m/%Y")

surveys_clean = s
print(surveys_clean.shape)
surveys_clean.head()

## Step 2b: clean Complaints

In [ ]:
c = complaints.copy()
c.columns = ["case_id", "customer_id", "open_date", "close_date", "route",
             "issue_type", "contact_count", "resolved"]

n = c.duplicated("case_id").sum()
c = c.drop_duplicates("case_id")
note("Complaints", "Duplicate CaseID", n, "Removed")

n = (c["route"] != c["route"].str.strip()).sum()
c["route"] = c["route"].str.strip()
note("Complaints", "Route with extra spaces", n, "Trimmed")

note("Complaints", "Issue type blank", c["issue_type"].isna().sum(), "Labelled Unknown")
c["issue_type"] = c["issue_type"].str.strip().str.lower().map(issue_map).fillna("Unknown")

bad = c["contact_count"] < 1
note("Complaints", "Contact count 0 or negative", bad.sum(), "Set to null")
c.loc[bad, "contact_count"] = np.nan

before = c["resolved"].copy()
c["resolved"] = c["resolved"].str.strip().str.upper().str[0].map({"Y": "Yes", "N": "No"})
note("Complaints", "Resolved spelling variants (Y, yes, YES)", (before != c["resolved"]).sum(), "Standardised")

c["open_date"]  = pd.to_datetime(c["open_date"],  format="%d/%m/%Y")
c["close_date"] = pd.to_datetime(c["close_date"], format="%d/%m/%Y")
note("Complaints", "Close date blank (case still open)", c["close_date"].isna().sum(), "Kept as null")

c["resolution_days"] = (c["close_date"] - c["open_date"]).dt.days

complaints_clean = c
print(complaints_clean.shape)
complaints_clean.head()

## Step 2c: clean Flights

In [ ]:
f = flights.copy()
f.columns = ["month", "route", "cabin_class", "passengers"]

n = f.duplicated().sum()
f = f.drop_duplicates()
note("Flights", "Exact duplicate rows", n, "Removed")

# Month: 'january', 'FEB', 'Mar.', 'APRIL' -> Jan, Feb, Mar, Apr
before = f["month"].copy()
f["month"] = f["month"].str.strip().str[:3].str.title()
note("Flights", "Month spelling variants", (before != f["month"]).sum(), "Standardised")
f["month_date"] = pd.to_datetime("2024-" + f["month"] + "-01", format="%Y-%b-%d")

# Passengers: text like 'unknown' and negative numbers -> null
f["passengers"] = pd.to_numeric(f["passengers"], errors="coerce")
bad = f["passengers"] < 0
note("Flights", "Negative passengers", bad.sum(), "Set to null")
f.loc[bad, "passengers"] = np.nan
note("Flights", "Passengers blank or text", f["passengers"].isna().sum() - bad.sum(), "Kept as null")

flights_clean = f[["month", "month_date", "route", "cabin_class", "passengers"]]
print(flights_clean.shape)
flights_clean.head()

## Step 3: data-quality log
This table is your **Data & approach** slide.

In [ ]:
quality_log = pd.DataFrame(log)
quality_log[quality_log["rows"] > 0]

## Step 4: load into PostgreSQL
Creates a schema called `pipeline`, so your earlier tables stay untouched.

In [ ]:
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.begin() as con:
    con.execute(text("DROP SCHEMA IF EXISTS pipeline CASCADE"))
    con.execute(text("CREATE SCHEMA pipeline"))

surveys_clean.to_sql("surveys", engine, schema="pipeline", index=False)
complaints_clean.to_sql("complaints", engine, schema="pipeline", index=False)
flights_clean.to_sql("flights", engine, schema="pipeline", index=False)
quality_log.to_sql("quality_log", engine, schema="pipeline", index=False)

# Routes dimension + keys (the star schema)
with engine.begin() as con:
    con.execute(text("""
        CREATE TABLE pipeline.routes AS
        SELECT route FROM pipeline.surveys
        UNION SELECT route FROM pipeline.complaints
        UNION SELECT route FROM pipeline.flights;
        ALTER TABLE pipeline.routes     ADD PRIMARY KEY (route);
        ALTER TABLE pipeline.surveys    ADD PRIMARY KEY (survey_id);
        ALTER TABLE pipeline.complaints ADD PRIMARY KEY (case_id);
        ALTER TABLE pipeline.surveys    ADD FOREIGN KEY (route) REFERENCES pipeline.routes(route);
        ALTER TABLE pipeline.complaints ADD FOREIGN KEY (route) REFERENCES pipeline.routes(route);
        ALTER TABLE pipeline.flights    ADD FOREIGN KEY (route) REFERENCES pipeline.routes(route);
    """))
print("Loaded into PostgreSQL")

## Step 5: check the load

In [ ]:
pd.read_sql("""
    SELECT 'surveys' AS table_name, COUNT(*) AS row_count FROM pipeline.surveys
    UNION ALL SELECT 'complaints', COUNT(*) FROM pipeline.complaints
    UNION ALL SELECT 'flights', COUNT(*) FROM pipeline.flights
    UNION ALL SELECT 'routes', COUNT(*) FROM pipeline.routes
""", engine)

## Step 6: quick KPI check in Python
Compare these with your SQL views and Power BI cards later. They should match.

In [ ]:
valid_csat = surveys_clean["csat_score"].dropna()
valid_nps  = surveys_clean["nps_score"].dropna()
fcr = ((complaints_clean["contact_count"] == 1) & (complaints_clean["resolved"] == "Yes")).mean()

print(f"CSAT %:  {(valid_csat >= 4).mean():.1%}")
print(f"NPS:     {((valid_nps >= 9).mean() - (valid_nps <= 6).mean()) * 100:.1f}")
print(f"Avg CES: {surveys_clean['ces_score'].mean():.2f}")
print(f"FCR %:   {fcr:.1%}")
print(f"Complaints per 1K pax: {len(complaints_clean) / flights_clean['passengers'].sum() * 1000:.2f}")